In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total number of character: {len(raw_text)}")
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
import re

# words = re.split(r'(\s)', raw_text)
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [word for word in preprocessed if word.strip()]

print(f"Total length of preprocessed wor: {len(preprocessed)}")
print(preprocessed[:99])

Total length of preprocessed wor: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter']


In [3]:
all_tokens = sorted(list(set(preprocessed)))
vocab_size = len(all_tokens)

print(f"Vocab length of dataset: {vocab_size}")

Vocab length of dataset: 1130


In [4]:
vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [5]:
for idx, item in enumerate(vocab.items()):
    if idx > 10:
        break
    
    print(item)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


In [6]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: item for item, idx in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [word for word in preprocessed if word.strip()]
        ids = [self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


        

In [7]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."

He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.
"""

encoded = tokenizer.encode(text)
print(f"Encoded Result: {encoded}", end="\n\n")

decoded = tokenizer.decode(encoded)
print(f"Decoded Result: {decoded}")

Encoded Result: [48, 607, 902, 5, 1109, 221, 5, 157, 805, 729, 722, 988, 320, 172, 467, 7, 1, 95, 8, 655, 1129, 278, 6, 157, 540, 169, 988, 264, 1126, 628, 7, 1, 48, 774, 990, 180, 697, 383, 157, 289, 1016, 1073, 1051, 157, 362, 988, 845, 5, 930, 714, 157, 991, 215, 988, 769, 7]

Decoded Result: He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward." There : make yourself comfortable -- and here are the cigars you like." He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.


In [8]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab_size = len(all_tokens)

print(f"Vocab length of dataset: {vocab_size}")

Vocab length of dataset: 1132


In [9]:
vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [10]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: item for item, idx in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [word for word in preprocessed if word.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


        

In [11]:
tokenizer = SimpleTokenizerV2(vocab)

text_1 = """
He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."

He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.
"""

text_2 = """
"How it happened? I can tell you in five minutes--and it didn't take much longer to happen. . . . I can remember now how surprised and pleased I was when I got Mrs. Stroud's note. Of course, deep down, I had always _felt_ there was no one like him--only I had gone with the stream, echoed the usual platitudes about him, till I half got to think he was a failure, one of the kind that are left behind. By Jove, and he _was_ left behind--because he had come to stay! The rest of us had to let ourselves be swept along or go under, but he was high above the current--on everlasting foundations, as you say.

"Well, I went off to the house in my most egregious mood--rather moved, Lord forgive me, at the pathos of poor Stroud's career of failure being crowned by the glory of my painting him! Of course I meant to do the picture for nothing--I told Mrs. Stroud so when she began to stammer something about her poverty. I remember getting off a prodigious phrase about the honour being _mine_--oh, I was princely, my dear Rickham! I was posing to myself like one of my own sitters.
"""


text = " <|endoftext|> ".join((text_1, text_2))
print(text, end="\n\n")

encoded = tokenizer.encode(text)
print(f"Encoded Result: {encoded}", end="\n\n")

decoded = tokenizer.decode(encoded)
print(f"Decoded Result: {decoded}")


He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."

He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.
 <|endoftext|> 
"How it happened? I can tell you in five minutes--and it didn't take much longer to happen. . . . I can remember now how surprised and pleased I was when I got Mrs. Stroud's note. Of course, deep down, I had always _felt_ there was no one like him--only I had gone with the stream, echoed the usual platitudes about him, till I half got to think he was a failure, one of the kind that are left behind. By Jove, and he _was_ left behind--because he had come to stay! The rest of us had to let ourselves be swept along or go under, but he was high above the current--on everlasting foundations, as you say.

"Well, I went off to the house in my most egregious mood--rather moved, Lord forgive me, at the pathos

In [12]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.n_vocab

50257

In [13]:
text_1 = """
He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."

He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.
"""

text_2 = """
"How it happened? I can tell you in five minutes--and it didn't take much longer to happen. . . . I can remember now how surprised and pleased I was when I got Mrs. Stroud's note. Of course, deep down, I had always _felt_ there was no one like him--only I had gone with the stream, echoed the usual platitudes about him, till I half got to think he was a failure, one of the kind that are left behind. By Jove, and he _was_ left behind--because he had come to stay! The rest of us had to let ourselves be swept along or go under, but he was high above the current--on everlasting foundations, as you say.

"Well, I went off to the house in my most egregious mood--rather moved, Lord forgive me, at the pathos of poor Stroud's career of failure being crowned by the glory of my painting him! Of course I meant to do the picture for nothing--I told Mrs. Stroud so when she began to stammer something about her poverty. I remember getting off a prodigious phrase about the honour being _mine_--oh, I was princely, my dear Rickham! I was posing to myself like one of my own sitters.
"""

text = "".join((text_1, text_2))
print(text, end="\n\n")

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)


He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."

He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.

"How it happened? I can tell you in five minutes--and it didn't take much longer to happen. . . . I can remember now how surprised and pleased I was when I got Mrs. Stroud's note. Of course, deep down, I had always _felt_ there was no one like him--only I had gone with the stream, echoed the usual platitudes about him, till I half got to think he was a failure, one of the kind that are left behind. By Jove, and he _was_ left behind--because he had come to stay! The rest of us had to let ourselves be swept along or go under, but he was high above the current--on everlasting foundations, as you say.

"Well, I went off to the house in my most egregious mood--rather moved, Lord forgive me, at the pathos of poor Stroud

In [14]:
strings = tokenizer.decode(integers)
strings

'\nHe laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."\n\nHe placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.\n\n"How it happened? I can tell you in five minutes--and it didn\'t take much longer to happen. . . . I can remember now how surprised and pleased I was when I got Mrs. Stroud\'s note. Of course, deep down, I had always _felt_ there was no one like him--only I had gone with the stream, echoed the usual platitudes about him, till I half got to think he was a failure, one of the kind that are left behind. By Jove, and he _was_ left behind--because he had come to stay! The rest of us had to let ourselves be swept along or go under, but he was high above the current--on everlasting foundations, as you say.\n\n"Well, I went off to the house in my most egregious mood--rather moved, Lord forgive me, at the pathos of p

In [15]:
text = "kibi kibi kaba kaba"

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.decode(integers)
print(strings)

for item in integers:
    print(f"{item}: {tokenizer.decode([item])}") 

[74, 27567, 479, 27567, 479, 15498, 479, 15498]
kibi kibi kaba kaba
74: k
27567: ibi
479:  k
27567: ibi
479:  k
15498: aba
479:  k
15498: aba


In [16]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)

print(f"Total number of character in raw_text: {len(raw_text)}")
print(raw_text[:99], end="\n\n")

print(f"Total number of character in enc_text: {len(enc_text)}")
print(enc_text[:99])

Total number of character in raw_text: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 

Total number of character in enc_text: 5145
[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 502, 284, 3285, 326, 11, 287, 262, 6001, 286, 465, 13476, 11, 339, 550, 5710, 465, 12036, 11, 6405, 257, 5527, 27075, 11, 290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561, 423, 587, 10598, 393, 28537, 2014, 198, 198, 1, 464, 6001, 286, 465, 13476, 1, 438, 5562, 373, 644, 262, 1466, 1444, 340, 13, 314, 460, 3285, 9074, 13, 46606]


In [17]:
text = "He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward."

enc_sample = tokenizer.encode(text)

context_size = 4 # length of the input
# the context size of 4 means that the model is trained to look a sequence of 4 words (or token)
# to predict the next word

X = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"X: {X}")
print(f"y:      {y}")

X: [1544, 13818, 4622, 11]
y:      [13818, 4622, 11, 1231]


In [18]:
for idx in range(1, context_size + 1):
    context = enc_sample[:idx]
    desired = enc_sample[1:idx + 1]

    print(f"{context} ----> {desired}")

[1544] ----> [13818]
[1544, 13818] ----> [13818, 4622]
[1544, 13818, 4622] ----> [13818, 4622, 11]
[1544, 13818, 4622, 11] ----> [13818, 4622, 11, 1231]


In [19]:
for idx in range(1, context_size + 1):
    context = enc_sample[:idx]
    desired = enc_sample[1:idx + 1]

    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode(desired)}")

He ---->  laughed
He laughed ---->  laughed slightly
He laughed slightly ---->  laughed slightly,
He laughed slightly, ---->  laughed slightly, without


In [20]:
import torch
from torch.utils.data import DataLoader, Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for idx in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[idx: idx + max_length]
            target_chunk = token_ids[idx + 1: idx + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [21]:
def create_dataloader_v1(
    txt,           # raw text string to tokenize and chunk
    batch_size=4,  # number of samples per batch
    max_length=256,# number of tokens in each input/target chunk (context window)
    stride=128,    # how many tokens to slide the window forward; overlap when stride < max_length
    shuffle=True,  # shuffle batches each epoch (disable for evaluation)
    drop_last=True,# drop the final incomplete batch so all batches are equal size
    num_workers=10 # parallel workers for data loading
):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False,
    num_workers=0
)

data_iter = iter(dataloader)
print(data_iter)

In [23]:
next_batch = next(data_iter)
print(next_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [24]:
next_batch = next(data_iter)
print(next_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
